# 音乐理解与智能混音助手 · Colab 特征提取流水线

**这个 notebook 只做一件事：下载 MTG-Jamendo 全量音频并提 MERT 特征。**
训练不在这里跑 —— 标签头只有 42 万参数，一次训练 96 秒，
本机跑比往返传数据快得多。Colab 的价值在于**数据中心带宽**和 GPU 前向。

### 为什么必须逐块流式处理

| | 体积 |
|---|---|
| 全量音频（100 块） | **~490 GB** |
| Colab 本地盘 | ~100–200 GB |
| 提出来的 4 段特征 | ~140 GB |

音频**塞不进本地盘**，所以流程必须是：
`下载一块 → 提特征 → 打包特征到云盘 → 删掉这块音频 → 下一块`。

### 为什么特征要打包成 tar 再传云盘

全量特征是 **5 万多个小 `.npy` 文件**。Google Drive 对大量小文件的写入极慢，
而且容易触发 API 配额。**每块打成一个 tar** → 100 个文件，稳定得多。

> ⚠️ **许可**：MTG-Jamendo 为 **非商业研究/学术用途**，音频与元数据**不可再分发**。
> 本 notebook 把数据下到你自己的云盘供你自己研究使用；**不要**把音频或特征公开分享。


## 0 · 挂载云盘并确认目录


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, shutil, subprocess, time

# 你在云盘里建好的目录（注意 'Audio AI' 中间有空格，所有路径都要加引号）
DRIVE = pathlib.Path('/content/drive/MyDrive/Audio AI/MusicMixer')
DRIVE.mkdir(parents=True, exist_ok=True)

# 特征包与元数据存云盘；音频只存本地盘（用完即删，永远不进云盘）
FEAT_DRIVE = DRIVE / 'features_tar'   ; FEAT_DRIVE.mkdir(exist_ok=True)
META_DRIVE = DRIVE / 'meta'           ; META_DRIVE.mkdir(exist_ok=True)
LOCAL      = pathlib.Path('/content/work')

free = shutil.disk_usage('/content').free / 2**30
print(f'云盘目录  : {DRIVE}')
print(f'本地可用盘: {free:.0f} GB')
assert free > 30, '本地盘不足 30 GB，无法容纳一块音频 + 特征'


## 1 · 拉取项目代码

**直接 clone 仓库，不在这里重写任何逻辑。**
notebook 里复制一份提特征代码看似方便，但两边会各自演化，
最后 Colab 出的特征和本机出的对不上 —— 而这种不一致是静默的。


In [ ]:
REPO = 'https://github.com/EthanBAI-dev/musicmix.git'
SRC  = pathlib.Path('/content/musicmix')

if SRC.exists():
    !cd {SRC} && git pull --ff-only
else:
    !git clone --depth 1 {REPO} {SRC}

%cd {SRC}
!git log --oneline -1


## 2 · 装依赖


In [ ]:
!pip -q install librosa soundfile pyloudnorm transformers torchaudio nnAudio

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), '没拿到 GPU：菜单 代码执行程序 → 更改运行时类型 → GPU'


## 3 · 配置

`SEGMENTS` 要和本机实验保持一致，否则数字不可比。
当前本机最佳配置是 **第 6 层 · 4 段**（P5 结论：4 段比中间 30 秒 +0.0268 mAP，5/5 种子一致）。


In [ ]:
LAYER      = 6
SEGMENTS   = 4          # 与本机一致；改这个数会让结果无法与既有实验比较
CHUNK_FROM = 10         # 本机已有 0-9，从 10 开始接着下
CHUNK_TO   = 100        # 全量共 100 块
SUBSET     = 'autotagging_top50tags'

# ---- 要不要把**音频**也留在云盘 ----
# False：音频用完即删，云盘只存特征（约 140 GB）。想换基座/换层重提时要重下。
# True ：音频也存云盘（约 490 GB），以后重提特征不用再下。5 TB 放得下。
#
# 存的是**下载时的分块 tar**，不是解开的 mp3。
# 全量是 5 万多个 mp3，往 Drive 里铺小文件极慢且易触发 API 配额；
# 每块一个 tar → 100 个文件，快且稳。
KEEP_AUDIO = True

DATA = SRC / 'data' / 'jamendo'
DATA.mkdir(parents=True, exist_ok=True)

AUDIO_DRIVE = DRIVE / 'audio_tar'
if KEEP_AUDIO:
    AUDIO_DRIVE.mkdir(exist_ok=True)
    print(f'音频 tar 将存到: {AUDIO_DRIVE}（预计 ~490 GB）')
else:
    print('音频用完即删，云盘只存特征（~140 GB）')

# 元数据只需一次，且很小，直接放云盘（重启会话不用重下）
if not (META_DRIVE / 'autotagging.tsv').exists():
    !python -m scripts.download_jamendo --meta-only --root {DATA}
    !cp -n {DATA}/meta/*.tsv '{META_DRIVE}/' 2>/dev/null || true
else:
    (DATA / 'meta').mkdir(exist_ok=True)
    !cp -n '{META_DRIVE}'/* {DATA}/meta/ 2>/dev/null || true
print('元数据文件:', len(list((DATA/'meta').glob('*'))))


## 4 · 主循环：下载 → 提特征 → 打包 → 删音频

**可以随时中断重跑。** 云盘上已存在 tar 的块会自动跳过，
所以 Colab 断开会话（12 小时上限）之后重新执行这一格即可接着走。


In [ ]:
# 特征目录名不要手写 —— 直接问代码要。
import sys, subprocess
sys.path.insert(0, str(SRC))
from src.tagging.backbone import BackboneConfig, MERT_95M

CFG = BackboneConfig(name=MERT_95M, layer=LAYER, n_segments=SEGMENTS)
FEAT_DIR = DATA / 'features' / CFG.tag()
print('特征目录:', FEAT_DIR)

# 循环里**不用 !shell 魔法**。
# 教训：`!tar -cf {tmp} -C {FEAT_DIR} ...` 里的 {} 替换没生效，
# tar 拿到字面量 '{FEAT_DIR}' 直接失败，而循环没检查返回码就往下走，
# 于是错误以「后面某步 FileNotFoundError」的形式冒出来，离真正的原因很远。
# subprocess 传列表：变量就是 Python 变量，没有任何字符串替换环节。
def run(cmd, what):
    r = subprocess.run([str(x) for x in cmd], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'   ❌ {what} 失败(码{r.returncode}):', (r.stderr or r.stdout)[-500:])
    return r.returncode == 0

def to_drive(local: pathlib.Path, dest: pathlib.Path) -> bool:
    """本地盘 → Drive。**必须 copy 而非 rename**：
    /content 与 /content/drive 是不同设备，os.rename 会抛 EXDEV。"""
    if not local.exists():
        print(f'   ❌ 源文件不存在: {local}'); return False
    shutil.copy2(local, dest)
    ok = dest.exists() and dest.stat().st_size == local.stat().st_size
    if ok: local.unlink()
    else:  print(f'   ❌ 复制到云盘后大小对不上: {dest}')
    return ok

# 每块**应有**多少首（只算所选子集）。用来判断本地特征是不是已经齐了 ——
# 只看「目录非空」是不够的：本项目吃过 tar 截断只解出 191/556 个文件、
# 而脚本据此永远跳过那一块的亏。
import collections
_want = collections.Counter()
with open(DATA/'meta'/f'{SUBSET}.tsv', encoding='utf-8') as _f:
    next(_f)
    for _line in _f:
        for _c in _line.rstrip('\n').split('\t'):
            if _c.endswith('.mp3'):
                _want[_c.split('/')[0]] += 1
                break

def feats_ready(i) -> bool:
    """本地是否已有这一块的完整特征（有就不用再下音频、不用再提）。"""
    sub = FEAT_DIR / f'{i:02d}'
    if not sub.exists():
        return False
    have, want = len(list(sub.rglob('*.npy'))), _want[f'{i:02d}']
    return want > 0 and have >= want * 0.98      # 容 2%：个别曲目解码失败是正常的

def feat_tar(i):  return FEAT_DRIVE / f'chunk{i:02d}_{CFG.tag()}.tar'
def audio_tar(i): return AUDIO_DRIVE / f'raw_30s_audio-{i:02d}.tar'
def audio_dir(i): return DATA / 'audio' / f'{i:02d}'
def local_tar(i): return DATA / 'audio' / f'_tmp_audio-{i:02d}.tar'

t_start = time.time()
for i in range(CHUNK_FROM, CHUNK_TO):
    if feat_tar(i).exists():
        print(f'[{i:02d}] 云盘已有特征包，跳过'); continue
    t0 = time.time()

    # 本地特征已经齐了（比如上次打包失败）→ **不重下、不重提**，直接打包
    if feats_ready(i):
        print(f'[{i:02d}] 本地已有完整特征，跳过下载与提取，直接打包')
    else:

        # (1) 拿音频：云盘已有 tar 就解包复用，否则下载
        if KEEP_AUDIO and audio_tar(i).exists():
            print(f'[{i:02d}] 云盘已有音频 tar，解包复用（不重下）')
            (DATA / 'audio').mkdir(parents=True, exist_ok=True)
            run(['tar', '-xf', audio_tar(i), '-C', DATA / 'audio'], '解包音频')
        else:
            cmd = [sys.executable, '-m', 'scripts.download_jamendo',
                   '--start', i, '--chunks', i + 1, '--root', DATA]
            if KEEP_AUDIO: cmd.append('--keep-tar')
            subprocess.run([str(x) for x in cmd], cwd=SRC)      # 下载要看进度，不吞输出
            if KEEP_AUDIO and local_tar(i).exists():
                to_drive(local_tar(i), audio_tar(i))

        if not audio_dir(i).exists():
            print(f'[{i:02d}] ⚠️ 音频缺失，跳过（重跑本格会自动重试）'); continue

        # (2) 提特征（已缓存的会跳过）。要看进度条，所以不吞输出
        subprocess.run([sys.executable, '-m', 'scripts.extract_backbone',
                        '--layer', str(LAYER), '--segments', str(SEGMENTS),
                        '--subset', SUBSET, '--root', str(DATA), '--device', 'cuda'], cwd=SRC)

    # (3) 打包特征 → 云盘。先本地打包再复制，且**校验**
    sub = FEAT_DIR / f'{i:02d}'
    if not sub.exists():
        print(f'[{i:02d}] ❌ 没有产出特征目录 {sub}，跳过打包'); continue
    n_npy = len(list(sub.rglob('*.npy')))
    tmp = pathlib.Path(f'/content/chunk{i:02d}.tar')
    if run(['tar', '-cf', tmp, '-C', FEAT_DIR, f'{i:02d}'], '打包特征'):
        if to_drive(tmp, feat_tar(i)):
            print(f'[{i:02d}] 已存云盘: {feat_tar(i).name}  ({n_npy} 个 npy)')

    # (4) 删本地音频腾地方（云盘上的 tar 不受影响）
    shutil.rmtree(audio_dir(i), ignore_errors=True)
    local_tar(i).unlink(missing_ok=True)

    el, tot = time.time()-t0, time.time()-t_start
    left = CHUNK_TO - i - 1
    free = shutil.disk_usage('/content').free / 2**30
    print(f'[{i:02d}] ✅ {el/60:.1f} min | 累计 {tot/3600:.1f} h | 本地剩 {free:.0f} GB | '
          f'剩 {left} 块，预计还要 {left*el/3600:.1f} h', flush=True)

print('\n🎉 全部完成')


## 5 · 校验：特征数量对不对得上元数据

**别只看「没报错」。** 本项目吃过亏：一次 tar 截断只解出 191/556 个文件，
而脚本只检查了「目录非空」，于是重跑永远跳过那一块。
这里逐块比对**实际文件数**与元数据里该块应有的曲目数。


In [ ]:
import tarfile, collections

# 元数据里每块应有多少首（只算 top50tags 子集）
want = collections.Counter()
with open(DATA/'meta'/f'{SUBSET}.tsv', encoding='utf-8') as f:
    next(f)
    for line in f:
        for c in line.rstrip('\n').split('\t'):
            if c.endswith('.mp3'):
                want[c.split('/')[0]] += 1
                break

bad = []
for i in range(CHUNK_FROM, CHUNK_TO):
    p = tar_path(i)
    if not p.exists():
        bad.append((f'{i:02d}', 'tar 缺失', 0, want[f'{i:02d}'])); continue
    with tarfile.open(p) as t:
        n = sum(1 for m in t.getmembers() if m.name.endswith('.npy'))
    w = want[f'{i:02d}']
    # 允许 2% 缺口：少数曲目解码失败是正常的，成片缺失才是问题
    if w and n < w * 0.98:
        bad.append((f'{i:02d}', '数量不足', n, w))

if bad:
    print('❌ 以下块有问题，删掉对应 tar 后重跑第 4 格：')
    for c, why, n, w in bad: print(f'   chunk {c}  {why}  {n}/{w}')
else:
    print(f'✅ chunk {CHUNK_FROM}–{CHUNK_TO-1} 全部通过校验')


## 6 · 取回本机

在**本机**终端跑（不是在 Colab 里）：

```bash
rclone copy gdrive:'Audio AI/MusicMixer/features_tar' ./tars --progress
```

或直接从 Drive 网页下载。然后解包到特征目录：

```bash
for f in tars/*.tar; do tar -xf "$f" -C data/jamendo/features/MERT-v1-95M_L6_s5_30s_x4/; done
```

**特征总量约 140 GB**，按需只取要用的块即可 —— 
先用一半验证结论是否随数据量变化，比一次拉满更稳妥。

### 之后在本机训练

```bash
python -m scripts.train_tagging --level L2 --arch attnhead --pooling mean \
    --layer 6 --segments 4 --seed 0 --out results/seeds/FULL_s0.json
```

**96 秒一次。** 不要为了这一步再回 Colab。
